In [1]:
import sys
from gitsource import GithubRepositoryDataReader

print(sys.executable)

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]

print("Lesson pages:", len(documents))
print("First filename:", documents[0]["filename"])

/opt/conda/envs/llm-zoomcamp/bin/python
Lesson pages: 72
First filename: 01-agentic-rag/lessons/01-intro.md


In [2]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query=query, num_results=5)

for rank, doc in enumerate(results, start=1):
    print(f"{rank}. {doc['filename']}")

1. 01-agentic-rag/lessons/14-agentic-loop.md
2. 01-agentic-rag/lessons/15-frameworks.md
3. 01-agentic-rag/lessons/13-function-calling.md
4. 01-agentic-rag/lessons/11-agents-intro.md
5. 01-agentic-rag/lessons/16-other-frameworks.md


In [3]:
context = "\n\n".join(
    f"Filename: {doc['filename']}\nContent:\n{doc['content']}"
    for doc in results
)

prompt = f"""
You are a course teaching assistant.
Answer the question using only the provided context.
If the context does not contain the answer, say "I don't know."

Question:
{query}

Context:
{context}
""".strip()

print("Retrieved pages:", len(results))
print("Prompt characters:", len(prompt))

Retrieved pages: 5
Prompt characters: 30092


In [4]:
from getpass import getpass
from openai import OpenAI

api_key = getpass("Paste your OpenAI API key: ")
client = OpenAI(api_key=api_key)
del api_key

print("Client ready")

Client ready


In [5]:
response_q3 = client.responses.create(
    model="gpt-5.4-mini",
    input=prompt,
)

print(response_q3.output_text)
print("\nInput tokens:", response_q3.usage.input_tokens)
print("Output tokens:", response_q3.usage.output_tokens)

The loop keeps calling the model by using a `while True` loop and checking whether the response contains any `function_call` items.

- It sets `has_function_calls = False` before each model call.
- It calls the model and appends the response to `messages`.
- If any item in `response.output` is a `function_call`, it runs the tool, appends the tool result, and sets `has_function_calls = True`.
- After processing the response, if `has_function_calls == False`, it breaks out of the loop.

So it stops when the model returns a response with no function calls, meaning it has given a final answer.

Input tokens: 7114
Output tokens: 138


In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print("Lesson pages:", len(documents))
print("Chunks:", len(chunks))
print("First chunk filename:", chunks[0]["filename"])
print("First chunk start:", chunks[0]["start"])
print("First chunk characters:", len(chunks[0]["content"]))

Lesson pages: 72
Chunks: 295
First chunk filename: 01-agentic-rag/lessons/01-intro.md
First chunk start: 0
First chunk characters: 2000


In [7]:
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(chunks)

chunk_results = chunk_index.search(query=query, num_results=5)

chunk_context = "\n\n".join(
    f"Filename: {doc['filename']}\nContent:\n{doc['content']}"
    for doc in chunk_results
)

chunk_prompt = f"""
You are a course teaching assistant.
Answer the question using only the provided context.
If the context does not contain the answer, say "I don't know."

Question:
{query}

Context:
{chunk_context}
""".strip()

response_q5 = client.responses.create(
    model="gpt-5.4-mini",
    input=chunk_prompt,
)

tokens_q3 = response_q3.usage.input_tokens
tokens_q5 = response_q5.usage.input_tokens

print(response_q5.output_text)
print("\nQ3 input tokens:", tokens_q3)
print("Q5 input tokens:", tokens_q5)
print(f"Reduction factor: {tokens_q3 / tokens_q5:.2f}×")

The loop keeps calling the model in a `while True` loop, and after each response it checks whether there were any `function_call` items. If there are function calls, it runs them, appends the results, and loops again. If a turn has no function calls, it breaks and stops.

Q3 input tokens: 7114
Q5 input tokens: 2297
Reduction factor: 3.10×


In [8]:
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Working folder:", Path.cwd())
print("Notebook file here:", Path("homework-01.ipynb").is_file())

for name in ("documents", "chunks", "client"):
    obj = globals().get(name)
    count = len(obj) if isinstance(obj, (list, tuple)) else "-"
    print(f"{name}: type={type(obj).__name__}, count={count}")

indexes = [
    (name, obj)
    for name, obj in list(globals().items())
    if type(obj).__module__.startswith("minsearch")
    and callable(getattr(obj, "search", None))
]

print("Index variables:", [name for name, _ in indexes])
for name, obj in indexes:
    records = getattr(obj, "docs", None)
    count = len(records) if isinstance(records, (list, tuple)) else "unknown"
    print(f"{name}: indexed records={count}")

Python: /opt/conda/envs/llm-zoomcamp/bin/python
Working folder: /workspaces/llmz-01-agentic-rag
Notebook file here: True
documents: type=list, count=72
chunks: type=list, count=295
client: type=OpenAI, count=-
Index variables: ['index', 'chunk_index']
index: indexed records=72
chunk_index: indexed records=295


In [12]:
%pip install toyaikit

  Using cached toyaikit-0.0.11-py3-none-any.whl.metadata (20 kB)
  Using cached anthropic-1.14.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached genai_prices-0.1.13-py3-none-any.whl.metadata (12 kB)
  Using cached docstring_parser-0.18.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached opentelemetry_api-1.45.1-py3-none-any.whl.metadata (1.4 kB)
Using cached toyaikit-0.0.11-py3-none-any.whl (24 kB)
Using cached anthropic-1.14.0-py3-none-any.whl (2.0 MB)
Using cached docstring_parser-0.18.0-py3-none-any.whl (22 kB)
Using cached opentelemetry_api-1.45.1-py3-none-any.whl (60 kB)
Using cached genai_prices-0.1.13-py3-none-any.whl (142 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [toyaikit]3/5 [anthropic]es]
Note: you may need to restart the kernel to use updated packages.


In [13]:
from importlib.metadata import version

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import (
    OpenAIResponsesRunner,
    DisplayingRunnerCallback,
)

print("ToyAIKit version:", version("toyaikit"))

ToyAIKit version: 0.0.11


In [14]:
search_calls = []

def search(query: str) -> list[dict]:
    """Search the course lesson chunks for passages relevant to the query."""
    search_calls.append(query)
    return chunk_index.search(query=query, num_results=5)

agent_tools = Tools()
agent_tools.add_tool(search)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lesson chunks for passages relevant to the query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [15]:
instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model="gpt-5.4-mini",
        client=client,
    ),
)

print("Agent configured.")

Agent configured.


In [16]:
search_calls.clear()

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

print("Search-tool calls:", len(search_calls))
for number, query in enumerate(search_calls, start=1):
    print(f"{number}. {query}")

-> Response received


-> Response received


Search-tool calls: 3
1. agentic loop RAG difference retrieval generation tool use course lesson
2. agentic loop how it works loop think act observe lesson
3. plain RAG vs agentic loop retrieval augmented generation course


### Q6 — Agent tool use

Run date: 15 October 2026 (Europe/Zurich).

- Framework: ToyAIKit 0.0.11
- Model: gpt-5.4-mini
- Retrieval: existing minsearch chunk_index, containing 295 chunks
- Question: How does the agentic loop work, and how is it different from plain RAG?
- Actual search-tool invocations: **3**
- Closest homework answer option: **4**

The first model response requested three searches with different queries.
After receiving their results, the next model response supplied the final answer.
The invocation count comes from the search_calls log.